# Leak-free preprocessing by hand

**Topic:** Fit, transform and leakage &nbsp;·&nbsp; **Difficulty:** Hard

A `ColumnTransformer` with an imputer, a scaler and a one-hot encoder learns a handful of
numbers from the training rows and applies them to every later row. Here you write those
two halves yourself with pandas and NumPy, on a small table of used bikes, then use them
inside cross-validation so that each fold learns only from its own training rows.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first. `train` holds the first eight bikes, and `new` holds two bikes that
arrive later, one with a brake type the shop has never listed.

In [ ]:
import numpy as np
import pandas as pd

# Used bikes listed by a resale shop. Gaps are NaN.
NUM = ["frame_cm", "km_ridden"]
CAT = ["brake"]
bikes = pd.DataFrame({
    "frame_cm":  [52, 56, np.nan, 54, 58, 61, 49, np.nan, 57, 50, 60, 68],
    "km_ridden": [1200, 300, 4500, np.nan, 800, 2500, 150, 3000, 900, np.nan, np.nan, 600],
    "brake":     ["rim", "disc", "rim", np.nan, "disc", "coaster", "rim", "disc",
                  "disc", "rim", "coaster", np.nan],
})
sold = np.array([1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1])   # sold within two weeks
train, new = bikes.iloc[:8], pd.DataFrame({
    "frame_cm": [np.nan, 50.0],
    "km_ridden": [600.0, np.nan],
    "brake": ["hydraulic", np.nan],
})
print(train)
print(new)

# The model for Part 3: predict the class whose mean training row is nearest.
def fit_centroids(Z, y):
    classes = np.unique(y)
    return classes, np.array([Z[y == c].mean(axis=0) for c in classes])

def predict_centroids(model, Z):
    classes, centers = model
    d = ((Z[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
    return classes[d.argmin(axis=1)]

## Part 1 (coding task): Learn the preprocessing from the training rows

Write `fit_prep(df, num_cols, cat_cols)`. It returns a dictionary `params` with four
dictionaries keyed by column name, holding what the fitted steps would store:

- `params["fill"][col]`: for a numeric column, the median of its known values. For a
  categorical column, its most frequent known value, with a tie going to the value that
  sorts first (as `SimpleImputer(strategy="most_frequent")` does).
- `params["mean"][col]` and `params["std"][col]`: for a numeric column, the mean and
  standard deviation that a `StandardScaler` placed after the imputer would learn. A spread
  of 0 becomes 1, as in `StandardScaler`.
- `params["categories"][col]`: for a categorical column, the sorted list of its values
  after the gaps are filled.

Use only the rows of `df`.

In [ ]:
def fit_prep(df, num_cols, cat_cols):
    """Learn every fill value, mean, spread and category list from df alone."""
    params = {"fill": {}, "mean": {}, "std": {}, "categories": {}}
    # TODO
    raise NotImplementedError

In [ ]:
p = fit_prep(train, NUM, CAT)
for key in ["fill", "mean", "std", "categories"]:
    assert key in p, f"params has no {key!r}"
assert np.isclose(p["fill"]["frame_cm"], 55.0), f"frame_cm fill {p['fill']['frame_cm']}: the median of the known training values"
assert np.isclose(p["fill"]["km_ridden"], 1200.0), f"km_ridden fill {p['fill']['km_ridden']}: the median of the known training values"
if np.isclose(p["mean"]["km_ridden"], 1778.5714, atol=1e-3) or np.isclose(p["std"]["frame_cm"], 3.9157800, atol=1e-4):
    raise AssertionError("the mean and spread come from the known values only: fit them on the column after filling its gaps")
assert np.isclose(p["mean"]["frame_cm"], 55.0) and np.isclose(p["mean"]["km_ridden"], 1706.25), f"means {p['mean']}"
if np.isclose(p["std"]["frame_cm"], 3.6253079, atol=1e-4):
    raise AssertionError("the spread uses the sample formula (ddof=1, the pandas default): StandardScaler divides by n")
assert np.isclose(p["std"]["frame_cm"], 3.3911650, atol=1e-4), f"frame_cm spread {p['std']['frame_cm']}"
assert np.isclose(p["std"]["km_ridden"], 1407.2219, atol=1e-2), f"km_ridden spread {p['std']['km_ridden']}"
assert p["fill"]["brake"] == "disc", f"brake fill {p['fill']['brake']!r}: rim and disc tie at 3, and a tie goes to the value that sorts first"
assert list(p["categories"]["brake"]) == ["coaster", "disc", "rim"], f"brake categories {list(p['categories']['brake'])}: the sorted training values, gaps filled first"
flat = fit_prep(pd.DataFrame({"frame_cm": [50.0, 50.0], "km_ridden": [1.0, 3.0], "brake": ["rim", "rim"]}), NUM, CAT)
assert flat["std"]["frame_cm"] == 1.0, "a column with no spread should get a spread of 1, as StandardScaler does"
print("Part 1 passed")

## Part 2 (coding task): Apply it to any rows

Write `transform_prep(params, df, num_cols, cat_cols)`. It returns a 2-D float array with
one row per row of `df`: first each numeric column, filled and standardized with the stored
numbers, then for each categorical column one 0/1 column per stored category, in the stored
order. A value that is not among the stored categories gets 0 in all of them, like
`OneHotEncoder(handle_unknown="ignore")`. The function must learn nothing from `df`.

In [ ]:
def transform_prep(params, df, num_cols, cat_cols):
    """Fill, scale and one-hot encode df with the stored params. Learns nothing."""
    # TODO
    raise NotImplementedError

In [ ]:
p = fit_prep(train, NUM, CAT)
Z_new = transform_prep(p, new, NUM, CAT)
print(np.round(Z_new, 4))
assert Z_new.shape == (2, 5), f"expected shape (2, 5): 2 numeric columns and 3 brake columns, got {Z_new.shape}"
expected = np.array([[0.0, -0.7861, 0.0, 0.0, 0.0],
                     [-1.4744, -0.3598, 0.0, 1.0, 0.0]])
assert not np.allclose(Z_new[1, :2], 0), "the new rows were scaled with their own statistics: use the ones in params"
assert np.allclose(Z_new[0, 2:], 0), "an unseen brake type should give all zeros in the brake columns"
assert np.allclose(Z_new, expected, atol=1e-3), "some values are wrong: fill, then subtract the stored mean and divide by the stored spread"
Z_train = transform_prep(p, train, NUM, CAT)
assert np.allclose(Z_train[:, :2].mean(axis=0), 0) and np.allclose(Z_train[:, :2].std(axis=0), 1), "the scaled training columns should have mean 0 and spread 1"
assert np.allclose(Z_train[:, 2:].sum(axis=1), 1), "each training row should have exactly one brake column set"
print("Part 2 passed")

## Part 3 (coding task): Cross-validate without a leak

Write `fold_accuracies(df, y, folds, num_cols, cat_cols)`. `folds` is a list of
`(train_idx, val_idx)` pairs of row positions. For each pair, learn the preprocessing with
`fit_prep` and fit the nearest-centroid model from the setup cell, then return the accuracy
on the validation rows. Return one accuracy per fold as a NumPy array. Call `fit_prep` and
`transform_prep` by name: the check watches which rows `fit_prep` is given.

In [ ]:
def fold_accuracies(df, y, folds, num_cols, cat_cols):
    """Accuracy of the nearest-centroid model on each (train_idx, val_idx) fold."""
    # TODO
    raise NotImplementedError

In [ ]:
folds = [(np.r_[4:12], np.r_[0:4]), (np.r_[0:4, 8:12], np.r_[4:8]), (np.r_[0:8], np.r_[8:12])]
seen = []
_fit_prep = fit_prep
def fit_prep(df, num_cols, cat_cols):
    seen.append(df)
    return _fit_prep(df, num_cols, cat_cols)
try:
    acc = fold_accuracies(bikes, sold, folds, NUM, CAT)
finally:
    fit_prep = _fit_prep

def rows_of(d):
    """The rows of a frame as sorted text, so the order and the index do not matter."""
    return sorted(map(str, d[NUM + CAT].to_numpy().tolist()))

print("fold accuracies:", np.round(acc, 3))
assert len(seen) > 0, "call fit_prep inside fold_accuracies, once per fold"
assert not any(len(d) == len(bikes) for d in seen), "fit_prep saw all 12 rows: fit it on each fold's training rows only"
assert len(seen) == len(folds), f"fit_prep should run once per fold, it ran {len(seen)} times"
for k, (d, (tr, va)) in enumerate(zip(seen, folds), start=1):
    assert rows_of(d) == rows_of(bikes.iloc[tr]), f"in fold {k}, fit_prep was given {len(d)} rows that are not the fold's {len(tr)} training rows"
assert np.shape(acc) == (3,), f"expected one accuracy per fold, got shape {np.shape(acc)}"
assert np.allclose(acc, [0.75, 1.0, 0.75]), f"fold accuracies {acc}"
print("Part 3 passed")

## Solution

The worked answer for each part, and the reason it is right. Every check above passes against these three functions.

### Part 1

**Why it works.** Each numeric statistic is learned in the order the pipeline runs its steps: the median
from the known values, then the mean and spread from the column with its gaps filled,
because the scaler only ever sees the imputer's output. NumPy's `std` divides by $n$, which
is what `StandardScaler` uses. The pandas method `Series.std` divides by $n - 1$ and would be wrong
here. For brakes, rim and disc tie at three listings each, and the tie goes to "disc".

In [ ]:
def fit_prep(df, num_cols, cat_cols):
    """Learn every fill value, mean, spread and category list from df alone."""
    params = {"fill": {}, "mean": {}, "std": {}, "categories": {}}
    for col in num_cols:
        fill = df[col].median()                    # NaN is skipped
        filled = df[col].fillna(fill).to_numpy(dtype=float)
        std = filled.std()                         # NumPy divides by n, as StandardScaler does
        params["fill"][col] = fill
        params["mean"][col] = filled.mean()
        params["std"][col] = std if std > 0 else 1.0
    for col in cat_cols:
        counts = df[col].value_counts()            # NaN is skipped
        ties = counts[counts == counts.max()].index
        fill = sorted(ties)[0]
        params["fill"][col] = fill
        params["categories"][col] = sorted(df[col].fillna(fill).unique())
    return params

### Part 2

**Why it works.** Every number used comes from `params`, so a row is transformed the same way whatever
else is in the batch. Comparing each value with the stored category list gives the one-hot
block in one broadcast, and a "hydraulic" brake matches no stored category, so its block
is all zeros. The second new bike's missing brake is filled with "disc" before encoding.

In [ ]:
def transform_prep(params, df, num_cols, cat_cols):
    """Fill, scale and one-hot encode df with the stored params. Learns nothing."""
    blocks = []
    for col in num_cols:
        x = df[col].fillna(params["fill"][col]).to_numpy(dtype=float)
        blocks.append(((x - params["mean"][col]) / params["std"][col])[:, None])
    for col in cat_cols:
        values = df[col].fillna(params["fill"][col]).to_numpy()
        cats = np.array(params["categories"][col], dtype=object)
        blocks.append((values[:, None] == cats[None, :]).astype(float))   # unseen: all zeros
    return np.hstack(blocks)

### Part 3

**Why it works.** The preprocessing is refit inside the loop on `df.iloc[train_idx]`, so the medians,
means, spreads and categories of each fold come from its training rows alone, which is what
`cross_val_score` does with a `Pipeline`. On these folds the accuracies are 0.75, 1 and
0.75, the same numbers scikit-learn's `ColumnTransformer` and `NearestCentroid` give in a
pipeline. Fitting `fit_prep` once on all 12 rows gives 0.75, 1 and 0.5 instead: the third
fold's 68 cm frame then helps set the frame mean and spread that its own rows are scaled
with. A leak can move a score either way, so the check watches the rows `fit_prep` is
given rather than trusting the scores alone.

In [ ]:
def fold_accuracies(df, y, folds, num_cols, cat_cols):
    """Accuracy of the nearest-centroid model on each (train_idx, val_idx) fold."""
    scores = []
    for train_idx, val_idx in folds:
        params = fit_prep(df.iloc[train_idx], num_cols, cat_cols)   # training rows only
        Z_tr = transform_prep(params, df.iloc[train_idx], num_cols, cat_cols)
        Z_val = transform_prep(params, df.iloc[val_idx], num_cols, cat_cols)
        model = fit_centroids(Z_tr, y[train_idx])
        scores.append(np.mean(predict_centroids(model, Z_val) == y[val_idx]))
    return np.array(scores)